# 🎤 Vocalido Chinese DiffSinger Training
**ทำทุกอย่างบน RunPod — ไม่ต้องโหลดลง Mac**

Pipeline:
1. Download M4Singer dataset (HuggingFace → RunPod)
2. Preprocess → DiffSinger format  
3. Train ต่อจาก `base_model.ckpt` (Chinese ✅)
4. Upload ckpt → GCS อัตโนมัติ

In [ ]:
# ── Cell 1: Environment Setup ─────────────────────────────────────────────────
import subprocess, os

def run(cmd, **kw):
    print(f'$ {cmd}')
    result = subprocess.run(cmd, shell=True, check=False, **kw)
    return result.returncode

# Install dependencies
run('pip install -q huggingface_hub datasets google-cloud-storage')
run('pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121')
run('apt-get install -qq -y libsndfile1 ffmpeg')
print('✅ Dependencies installed')

In [ ]:
# ── Cell 2: GCS Authentication ───────────────────────────────────────────────
# วาง Service Account JSON ของ vocamind@gmail.com ที่นี่
GCS_BUCKET = 'gs://vocalido-master-corpus-v1'
GCS_KEY_PATH = '/tmp/gcs_key.json'

# *** วาง Service Account JSON ลงในช่อง secret ของ RunPod ***
# หรือ paste ที่นี่:
GCS_KEY_JSON = '''{
  PASTE_YOUR_SERVICE_ACCOUNT_JSON_HERE
}'''

with open(GCS_KEY_PATH, 'w') as f:
    f.write(GCS_KEY_JSON)

os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = GCS_KEY_PATH
run(f'gcloud auth activate-service-account --key-file={GCS_KEY_PATH}')
run(f'gsutil ls {GCS_BUCKET}')
print('✅ GCS authenticated')

In [ ]:
# ── Cell 3: Clone DiffSinger + Download base_model from GCS ──────────────────
WORK_DIR = '/workspace'
DS_DIR   = f'{WORK_DIR}/DiffSinger'

if not os.path.exists(DS_DIR):
    run(f'git clone --depth 1 https://github.com/openvpi/DiffSinger.git {DS_DIR}')

os.makedirs(f'{DS_DIR}/checkpoints/base_model', exist_ok=True)
os.makedirs(f'{DS_DIR}/checkpoints/vocalido_chinese', exist_ok=True)

# Download base_model from GCS (อัปโหลดไว้แล้วจากเซสชั่นก่อน)
run(f'gsutil -m cp {GCS_BUCKET}/checkpoints/base_model.ckpt {DS_DIR}/checkpoints/base_model/')

# ถ้ายังไม่มีใน GCS ให้ดาวน์โหลดจาก OpenVPI HuggingFace
import os.path as osp
if not osp.exists(f'{DS_DIR}/checkpoints/base_model/base_model.ckpt'):
    print('Downloading base_model from OpenVPI HuggingFace...')
    run('pip install -q huggingface_hub')
    run(f'''python3 -c "
from huggingface_hub import hf_hub_download
hf_hub_download(
    repo_id='openvpi/DiffSinger',
    filename='base_model/model_ckpt_steps_160000.ckpt',
    local_dir='{DS_DIR}/checkpoints/base_model',
    local_dir_use_symlinks=False
)
"''')

print('✅ DiffSinger cloned, base_model ready')

In [ ]:
# ── Cell 4: Download M4Singer Dataset (Chinese, 20 singers) ──────────────────
# M4Singer = ~700MB compressed, 20 Mandarin Chinese singers
# ไม่ต้อง register, ดาวน์โหลดได้ตรงจาก HuggingFace

DATA_RAW  = f'{WORK_DIR}/m4singer_raw'
DATA_PROC = f'{DS_DIR}/data/raw/m4singer'
os.makedirs(DATA_RAW, exist_ok=True)

run(f'''python3 -c "
from huggingface_hub import snapshot_download
snapshot_download(
    repo_id='m-a-p/M4Singer',
    repo_type='dataset',
    local_dir='{DATA_RAW}',
    local_dir_use_symlinks=False,
    ignore_patterns=['*.parquet']  # skip heavy parquet, use audio files
)
print('M4Singer downloaded!')
"''')

print('\n✅ M4Singer download complete')
run(f'du -sh {DATA_RAW}')

In [ ]:
# ── Cell 5: Preprocess M4Singer → DiffSinger format ──────────────────────────
import json, csv, glob, shutil
import librosa, soundfile as sf
import numpy as np
from pathlib import Path

SR = 44100
PROC_DIR = Path(f'{WORK_DIR}/vocalido_chinese_dataset')
WAV_DIR  = PROC_DIR / 'wavs'
WAV_DIR.mkdir(parents=True, exist_ok=True)

transcriptions = []   # will be written to transcriptions.csv
raw_root = Path(DATA_RAW)

# M4Singer structure: {singer}/{song}/{utterance}.wav + .txt
wav_files = sorted(raw_root.rglob('*.wav'))[:500]  # max 500 files for first run

print(f'Processing {len(wav_files)} wav files...')

for i, wav_path in enumerate(wav_files):
    try:
        # Load and resample to 44100Hz
        audio, orig_sr = librosa.load(str(wav_path), sr=SR, mono=True)
        duration = len(audio) / SR
        if duration < 0.5 or duration > 20.0:  # skip too short/long
            continue

        # Find corresponding label file
        label_path = wav_path.with_suffix('.txt')
        if not label_path.exists():
            label_path = wav_path.with_name(wav_path.stem + '_label.txt')

        # Output wav
        out_name = f'chinese_{i:05d}'
        out_wav  = WAV_DIR / f'{out_name}.wav'
        sf.write(str(out_wav), audio, SR, subtype='PCM_24')

        # Read label (M4Singer label = pinyin text per utterance)
        if label_path.exists():
            label_text = label_path.read_text(encoding='utf-8').strip()
        else:
            label_text = '1'  # fallback to Yi

        transcriptions.append({
            'name': out_name,
            'ph_seq': label_text,   # will be converted in next step
            'ph_dur': str(round(duration / max(1, len(label_text.split())), 3)),
            'note_seq': 'C4',
            'note_dur': str(round(duration, 3)),
            'note_slur': '0',
            'f0_timestep': '0.01159',
        })

        if i % 50 == 0:
            print(f'  [{i}/{len(wav_files)}] {out_name}')

    except Exception as e:
        print(f'  ⚠️  Skip {wav_path.name}: {e}')
        continue

# Write transcriptions.csv
csv_path = PROC_DIR / 'transcriptions.csv'
with open(csv_path, 'w', newline='', encoding='utf-8') as f:
    fieldnames = ['name','ph_seq','ph_dur','note_seq','note_dur','note_slur','f0_timestep']
    w = csv.DictWriter(f, fieldnames=fieldnames)
    w.writeheader()
    w.writerows(transcriptions)

print(f'\n✅ Preprocessed {len(transcriptions)} files → {PROC_DIR}')

In [ ]:
# ── Cell 6: Write Chinese Training Config ────────────────────────────────────
import yaml

config = {
    # ── Base model (Chinese trained) ──────────────────────────────────────────
    'base_config': 'configs/acoustic/base.yaml',
    'finetune_ckpt_path': 'checkpoints/base_model/base_model.ckpt',

    # ── DO NOT freeze phoneme embeddings (Chinese → learn from data) ──────────
    # Empty = train ALL layers including embed_tokens
    'finetune_ignored_params': [],
    'finetune_strict_shapes': False,  # allow vocab size mismatch

    # ── Dataset ───────────────────────────────────────────────────────────────
    'raw_data_dir': str(PROC_DIR),
    'binary_data_dir': 'data/binary/vocalido_chinese',
    'dictionary': 'dictionaries/opencpop-extension.txt',  # Chinese phonemes ✅
    'num_spk': 1,
    'use_spk_id': False,

    # ── Audio ─────────────────────────────────────────────────────────────────
    'audio_sample_rate': 44100,
    'hop_size': 512,
    'fft_size': 2048,
    'win_size': 2048,
    'fmin': 40,
    'fmax': 16000,
    'num_mels': 128,

    # ── Training ──────────────────────────────────────────────────────────────
    'max_updates': 60000,          # 60k steps is plenty with Chinese base
    'val_check_interval': 2000,
    'num_ckpt_keep': 3,
    'lr': 0.0002,
    'lr_scheduler': 'step',
    'optimizer_adam_beta1': 0.9,
    'optimizer_adam_beta2': 0.98,
    'batch_size': 16,
    'max_batch_frames': 80000,
    'max_batch_size': 48,
    'num_workers': 4,

    # ── Model ─────────────────────────────────────────────────────────────────
    'hidden_size': 256,
    'num_heads': 2,
    'enc_layers': 4,
    'enc_ffn_kernel_size': 9,
    'dec_layers': 4,
    'use_shallow_diffusion': True,
    'K_step': 1000,
    'K_step_infer': 20,

    # ── Exp name ──────────────────────────────────────────────────────────────
    'exp_name': 'vocalido_chinese',
}

config_path = f'{DS_DIR}/configs/vocalido_chinese.yaml'
os.makedirs(os.path.dirname(config_path), exist_ok=True)
with open(config_path, 'w') as f:
    yaml.dump(config, f, default_flow_style=False, allow_unicode=True)

print(f'✅ Config written → {config_path}')
print(yaml.dump(config, default_flow_style=False))

In [ ]:
# ── Cell 7: Binarize Dataset ─────────────────────────────────────────────────
import subprocess
os.chdir(DS_DIR)

result = subprocess.run(
    'PYTHONPATH=. python data_gen/tts/bin/binarize.py '
    '--config configs/vocalido_chinese.yaml',
    shell=True, capture_output=True, text=True
)
print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[-2000:])
    raise RuntimeError('Binarize failed')
print('✅ Dataset binarized')

In [ ]:
# ── Cell 8: TRAIN ─────────────────────────────────────────────────────────────
# ใช้เวลา ~4-6 ชั่วโมงบน RTX 5090 สำหรับ 60k steps
print('🚀 Starting Chinese DiffSinger training...')
print('Expected: ~4-6 hours on RTX 5090')

train_cmd = (
    'PYTHONPATH=. python tasks/run.py '
    '--config configs/vocalido_chinese.yaml '
    '--exp_name vocalido_chinese '
    '--reset 2>&1 | tee /tmp/training_log.txt'
)

# Run training (blocking)
os.system(train_cmd)

In [ ]:
# ── Cell 9: Upload Best Checkpoint to GCS ────────────────────────────────────
import glob

ckpt_dir = f'{DS_DIR}/checkpoints/vocalido_chinese'
ckpts = sorted(glob.glob(f'{ckpt_dir}/*.ckpt'))

if ckpts:
    best_ckpt = ckpts[-1]  # latest = highest step
    ckpt_name = os.path.basename(best_ckpt)
    gcs_dest = f'{GCS_BUCKET}/checkpoints/chinese/{ckpt_name}'

    print(f'Uploading {ckpt_name} → GCS...')
    ret = run(f'gsutil -m cp {best_ckpt} {gcs_dest}')
    if ret == 0:
        print(f'✅ Checkpoint uploaded: {gcs_dest}')
        print(f'\nดาวน์โหลดลง Mac ด้วย:')
        print(f'gsutil cp {gcs_dest} ~/vocamind-projects/Memolody_V2/vocalido_server/training/DiffSinger/checkpoints/vocalido_chinese/')
    else:
        print('❌ Upload failed — check GCS credentials')
else:
    print('❌ No checkpoints found. Check training log.')
    run('tail -50 /tmp/training_log.txt')

In [ ]:
# ── Cell 10 (Optional): Upload config + dictionary to GCS ────────────────────
run(f'gsutil cp {config_path} {GCS_BUCKET}/checkpoints/chinese/config.yaml')
print('✅ Config uploaded. Chinese model is ready!')